<div align="center">

# 🧠 Digit Predictor: Backpropagation vs Predictive Coding
### The app's own network, trained the old way and the brain-like way, side by side

</div>

This notebook is a research companion to my paper *Beyond Attention and State* (RD-S4). Before changing how RD-S4 learns, i
test the change on the smallest network we have: the digit predictor (original app:
https://github.com/S4nfs/NN-Digit-Predictor; its PyTorch port `digit_predictor_pytorch.ipynb` is left untouched).
The network, the training data and the drawing board are the same. Only the learning rule changes. The digit
predictor is a plain multilayer perceptron, so there is no attention to replace here: the change is entirely in
how the weights learn.

**What i found.** Every number below is measured further down in this notebook. Values are means over seeds, and
p-values are exact permutation tests over all relabellings of the per-seed results.

| Question | Answer | Evidence |
|---|---|---|
| Can this network learn without backpropagation? | **Yes** | Predictive coding (PC) reaches 97.60% on real handwriting (MNIST) against backprop's 97.83% (3 seeds, p = 0.20), while executing zero backward kernels. After 2 to 5 relaxation steps its purely local updates point the same way as backprop's gradient (cosine 0.97 to 1.00 on a trained network). |
| Is it faster by itself? | **No** | On the GPU, PC needs 13.0 s for the 8 epochs backprop does in 10.1 s. The fully parallel version (iPC) needs 53.7 s and 10x the arithmetic. Every layer really does update at once, but a GPU turns that into more kernel launches, not more speed. On the app's own task PC needs fewer epochs (4.0 against 5.2, p = 0.024), at the same CPU time. |
| What made training faster? | **Learning only from surprise** | SPOC, our variant, matches backprop (97.87% against 97.83%, p = 0.80) and reaches 97.5% in 2.3 s instead of 6.2 s, because after the first epoch it learns from only 13.5% of the samples. Backprop with the same gate does just as well (2.1 s), so the speed-up belongs to the gate, which works with either learning rule. |
| Can it learn new drawings without forgetting? | **Yes, with orthogonal consolidation** | Taught 200 real digits one at a time, backprop fine-tuning at its best setting kept 70% of the app's original digits and 72% of the taught ones. With the app's own optimiser it collapsed to 15%. SPOC kept 100% of the original digits and 95% of the taught ones, while learning nearly as much (54.7% against 58.7% on MNIST, p = 0.30). Backprop errors sent through the same consolidation also keep 100%, so the protection comes from the consolidation; SPOC gets it from local signals only. |
| Does it survive depth? | **Not yet** | PC matches backprop with 2 to 4 layers, but trails by 2.3 points with 8 plain layers and with 16 residual blocks, while taking 5 to 6 times longer. This is the main obstacle for RD-S4. |
| Is inference faster? | **Unchanged** | Both trained networks answer with the same single feedforward sweep. |



## 1. Setup

Installs `torch`, `torchvision`, `gradio` and `matplotlib` only if the active kernel does not have them. Every
benchmark saves its results under `benchmarks/predictive_coding/results/`; re-running the notebook loads them in
seconds. Set `RERUN = True` in section 2 to measure everything again (about 15 minutes, most of it on the GPU).

In [1]:
import importlib.util
import subprocess
import sys

REQUIREMENTS = {"torch": "torch", "torchvision": "torchvision", "gradio": "gradio>=5", "matplotlib": "matplotlib"}
missing = [spec for module, spec in REQUIREMENTS.items() if importlib.util.find_spec(module) is None]
if not missing:
    print("✅ All dependencies are already installed.")
elif importlib.util.find_spec("pip") is None:
    raise ModuleNotFoundError(f"Missing {', '.join(missing)} and pip is not available in this kernel's environment.")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    importlib.invalidate_caches()
    print(f"Installed {', '.join(missing)}.")

✅ All dependencies are already installed.


## 2. From backpropagation to predictive coding

**What backpropagation does.** A forward pass computes every layer in turn. A global loss compares the output with
the label. The backward pass then carries that one error down the network, one layer after another, through the
chain rule, while every neuron holds on to the activity it had during the forward pass. Only then do the weights
change. The paper (`Brain Based Algorithms instead of Backpropagation`) names the two properties
of this procedure that a brain cannot have: separate phases that freeze activity (discontinuous processing), and a
strict layer-by-layer order coordinated from outside (no local autonomy).

**What predictive coding does instead** (Rao and Ballard 1999; Whittington and Bogacz 2017; Bogacz 2017). Every
layer $l$ is predicted from the layer below it:

$$\mu^l = W^l f(x^{l-1}) + b^l, \qquad \varepsilon^l = x^l - \mu^l .$$

Each layer has value neurons $x^l$ and error neurons $\varepsilon^l$. The network has an energy, its total
prediction error,

$$\mathcal{F} = \sum_{l=1}^{L-1} \tfrac{1}{2}\lVert \varepsilon^l \rVert^2 + \mathrm{CE}\big(\mu^L, y\big),$$

where the last term is the app's own categorical cross-entropy, so the output error is
$\varepsilon^L = y - \mathrm{softmax}(\mu^L)$. Clamp the pixels at the bottom and the label at the top, and let
every value neuron roll downhill on $\mathcal{F}$ at the same time. This is the equation on the paper's slide,
with the nonlinearity kept:

$$\frac{dx^l}{dt} = -\varepsilon^l + f'(x^l) \odot (W^{l+1})^\top \varepsilon^{l+1}.$$

A neuron needs only its own error and the errors of the layer it predicts. When the activities have settled, each
synapse changes by the product of the two signals it can physically see, its post-synaptic error and its
pre-synaptic activity (a Hebbian rule):

$$\Delta W^l \propto \varepsilon^l f(x^{l-1})^\top, \qquad \Delta b^l \propto \varepsilon^l .$$

No backward pass, chain rule or autograd appears anywhere in the predictive coding code below. The kernel census at
the end of section 5 checks this: a predictive coding step executes zero backward kernels.

| | What updates, and when | Reference |
|---|---|---|
| **BP** | forward pass, global loss, backward pass layer by layer, then all weights | the app, unchanged |
| **PC** | all activities relax together for $T$ steps, then all synapses update together | Whittington and Bogacz (2017) |
| **iPC** | activities and synapses all update together at every step: no phases at all | Salvatori et al. (2024) |
| **SPOC** | PC, with plasticity gated by surprise and new knowledge consolidated orthogonally (section 6) | this notebook |
| **BP + surprise** | backprop with the same surprise gate as SPOC: a control | Katharopoulos and Fleuret (2018) |

One caveat from the paper also applies here: the top-down error uses the transpose of the bottom-up weights (the
weight transport problem). Like the predictive coding literature, we keep the weights symmetric.

The next cell holds every setting in one place. All randomness is seeded.

In [2]:
import copy
import json
import math
import random
import subprocess
import time
from itertools import combinations
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display

CPU = torch.device("cpu")
GPU = torch.device("cuda") if torch.cuda.is_available() else CPU
torch.set_num_threads(4)              # fixed, so CPU timings are comparable between runs

WORK = Path("benchmarks/predictive_coding")
DATA_DIR, RESULTS_DIR = WORK / "data_cache", WORK / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RERUN = False                         # True recomputes every benchmark instead of loading saved results

SIZES = (784, 128, 64, 10)            # the app's network: 784 -> 128 -> 64 -> 10
APP = dict(epochs=45, batch=16, lr=0.01, lr_ipc=1e-3, seeds=[0, 1, 2, 3, 4])
MN = dict(epochs=8, batch=64, seeds=[0, 1, 2], sweep_epochs=2, lrs=[1e-3, 3e-3, 1e-2], ipc_lrs=[1e-4, 3e-4, 1e-3])
PCFG = dict(T=5, gamma=0.1, T_ipc=10, tau=0.1, p_min=0.05)
DREAM = dict(stream=200, cycles=5, every=20, seeds=[0, 1, 2], alpha=1e-2, eta=1.0, T=20, gamma=0.1,
             bp_lr=1e-3, app_lr=0.01, pc_lr=1e-3)
DEPTH = dict(epochs=2, batch=64, lr=3e-3, plain=[2, 4, 6, 8], residual=[2, 4, 8, 16], seed=0)

METHODS = ("BP", "BP + surprise", "PC", "iPC", "SPOC")
GATED = ("BP + surprise", "SPOC")
COLORS = {"BP": "tab:blue", "BP + surprise": "#7fb3e6", "PC": "tab:green", "iPC": "tab:orange", "SPOC": "black"}
STYLE = {"BP": "-", "BP + surprise": "--", "PC": "-", "iPC": "-", "SPOC": "-"}
TAG = "[BP | PC | iPC | SPOC]"

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.constrained_layout.use": True})


def cached(name, config, compute):
    """Load results/<name>.json if it was produced with this exact config, otherwise compute and save it."""
    config = json.loads(json.dumps(config))
    path = RESULTS_DIR / f"{name}.json"
    if path.exists() and not RERUN:
        saved = json.loads(path.read_text())
        if saved.get("config") == config:
            print(f"[{name}] loaded saved results ({saved['seconds']:.0f}s to compute; set RERUN = True to redo)")
            return saved["result"]
    t0 = time.perf_counter()
    result = compute()
    seconds = time.perf_counter() - t0
    path.write_text(json.dumps({"config": config, "result": result, "seconds": seconds}))
    print(f"[{name}] computed in {seconds:.0f}s")
    return result


def sync(device):
    """CUDA runs asynchronously: without this a timer measures kernel launch, not execution."""
    if device.type == "cuda":
        torch.cuda.synchronize()


def gpu_info():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=name,clocks.sm,temperature.gpu,power.draw,memory.used",
                               "--format=csv,noheader"], capture_output=True, text=True, timeout=10).stdout.strip()
    except Exception:  # noqa: BLE001
        return "n/a"


def perm_test(a, b, alternative="two-sided"):
    """Exact permutation test on the difference of means over all relabellings of the per-seed values."""
    a, b = [float(v) for v in a], [float(v) for v in b]
    pooled, n, obs = np.array(a + b), len(a), np.mean(a) - np.mean(b)
    hits = total = 0
    for idx in combinations(range(len(pooled)), n):
        mask = np.zeros(len(pooled), bool)
        mask[list(idx)] = True
        d = pooled[mask].mean() - pooled[~mask].mean()
        hits += {"two-sided": abs(d) >= abs(obs) - 1e-12, "less": d <= obs + 1e-12, "greater": d >= obs - 1e-12}[alternative]
        total += 1
    return hits / total


def mean_sd(values):
    v = np.asarray(values, float)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)


print(f"PyTorch {torch.__version__} | CPU threads {torch.get_num_threads()} | GPU: {gpu_info() if GPU.type == 'cuda' else 'none'}")


PyTorch 2.11.0+cu128 | CPU threads 4 | GPU: NVIDIA GeForce RTX 5050 Laptop GPU, 1597 MHz, 65, 39.87 W, 3500 MiB
